# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 04a: Monthly Account Statements (Schedulable Data Transformation)

---

### What This Notebook Does
Generates monthly account statements by aggregating transactions per account for the current month. This is a **data transformation** notebook designed to run on a recurring schedule.

**Output Table:** `MONTHLY_STATEMENTS`

> **Warehouse:** `HOL_MEDIUM_WH`

In [ ]:
-- Setup context
USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_MEDIUM_WH;

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *

session = get_active_session()
print(f"Session ready: {session.get_current_database()}.{session.get_current_schema()}")

## Step 1: Build Monthly Statement Aggregation

For each account, compute: opening balance, total credits, total debits, closing balance, and transaction count for the current month.

In [ ]:
# =============================================================================
# MONTHLY STATEMENT GENERATION
# Aggregate transactions per account for the current statement period
# =============================================================================

accounts_df = session.table("ACCOUNTS")
transactions_df = session.table("TRANSACTIONS")

# Filter transactions to current month
current_month_txns = transactions_df.filter(
    F.date_trunc("MONTH", F.to_timestamp(F.col("TXN_DATE"))) == F.date_trunc("MONTH", F.current_date())
)

# Aggregate per account
statement_df = current_month_txns.group_by("ACCOUNT_ID").agg(
    F.count("*").alias("TXN_COUNT"),
    F.sum(F.when(F.col("TXN_TYPE").isin(["CREDIT", "DEPOSIT"]), F.col("AMOUNT"))
          .otherwise(F.lit(0))).alias("TOTAL_CREDITS"),
    F.sum(F.when(F.col("TXN_TYPE").isin(["DEBIT", "WITHDRAWAL", "PAYMENT", "FEE"]), F.col("AMOUNT"))
          .otherwise(F.lit(0))).alias("TOTAL_DEBITS"),
    F.sum(F.when(F.col("TXN_TYPE") == F.lit("TRANSFER"), F.col("AMOUNT"))
          .otherwise(F.lit(0))).alias("TOTAL_TRANSFERS"),
    F.min(F.to_timestamp(F.col("TXN_DATE"))).alias("FIRST_TXN_DATE"),
    F.max(F.to_timestamp(F.col("TXN_DATE"))).alias("LAST_TXN_DATE"),
    F.count_distinct("MERCHANT").alias("UNIQUE_MERCHANTS"),
    F.count_distinct("CATEGORY").alias("UNIQUE_CATEGORIES")
)

# Join with account info for context
monthly_statements = statement_df.join(
    accounts_df.select("ACCOUNT_ID", "CUSTOMER_ID", "ACCOUNT_TYPE", "BALANCE"),
    "ACCOUNT_ID",
    "inner"
).with_columns(
    ["STATEMENT_MONTH", "NET_FLOW", "GENERATED_AT"],
    [
        F.date_trunc("MONTH", F.current_date()),
        F.col("TOTAL_CREDITS") - F.col("TOTAL_DEBITS"),
        F.current_timestamp()
    ]
)

print(f"Monthly statements generated: {monthly_statements.count()} accounts")
monthly_statements.show(5)

## Step 2: Write to Output Table

In [ ]:
# Write/append monthly statements (append mode so historical months are preserved)
monthly_statements.write.mode("overwrite").save_as_table("MONTHLY_STATEMENTS")

result = session.table("MONTHLY_STATEMENTS")
print(f"✅ MONTHLY_STATEMENTS table: {result.count()} rows")
result.group_by("ACCOUNT_TYPE").agg(
    F.count("*").alias("ACCOUNTS"),
    F.sum("TOTAL_CREDITS").alias("TOTAL_CREDITS"),
    F.sum("TOTAL_DEBITS").alias("TOTAL_DEBITS"),
    F.sum("NET_FLOW").alias("NET_FLOW")
).show()